# Learning from Preference, and Measuring Whether It Worked

Supervised learning shows a model the right answer. **Reinforcement learning** only tells it *how good* the answer it produced was — which is the only feedback available when there is no single right answer, as with "write a helpful reply". This lab builds that machinery from arithmetic: a gradient measured with a ruler, an explore/exploit trade-off, REINFORCE, PPO's clip, and a complete DPO trainer.

Then the harder half. Every method here optimises a *number*, and the number is never quite what you meant. So the second half builds the measuring instruments: pass@k with its unbiased estimator, an eval harness with honest confidence intervals, and a judge whose biases we implement on purpose so we can measure them and then remove them.

All of it is numpy on tiny arrays. What is toy is the scale — ten parameters instead of seven billion. What is faithful is every update rule and every loss.

**How to use this notebook:** run cells top to bottom; later sections reuse earlier functions. Companion reading: Chapters 31, 32 and 33.

## 1. A gradient is a measured slope

Before any learning, one idea: the **gradient** of a loss with respect to a parameter is just "how much does the loss change per unit of parameter, right here". You can measure it with a ruler — nudge the parameter by a tiny $h$ in each direction and take rise over run:

$$
\frac{dL}{dw} \approx \frac{L(w + h) - L(w - h)}{2h}
$$

That is a **finite difference**. It needs no calculus, it works on any function you can call, and — this is the point of the cell below — it is how you *check* an analytic gradient you derived by hand. Every sign error you will ever make is caught here, in one second, for twenty extra function calls.

In [ ]:
import numpy as np

def loss(w):
    return (w - 3.0) ** 2 + 2.0

def numeric_slope(f, w, h=1e-6):
    """Rise over run: how much does f change per unit of w, right here?"""
    return (f(w + h) - f(w - h)) / (2 * h)

def analytic_slope(w):
    """d/dw [(w - 3)^2 + 2] = 2(w - 3), derived by hand."""
    return 2.0 * (w - 3.0)

print(f"{'w':>6}{'L(w)':>10}{'numeric':>11}{'analytic':>11}{'disagreement':>14}")
worst = 0.0
for w in [0.0, 2.0, 3.0, 5.0, 7.5]:
    num, ana = numeric_slope(loss, w), analytic_slope(w)
    worst = max(worst, abs(num - ana))
    print(f"{w:>6.1f}{loss(w):>10.4f}{num:>11.5f}{ana:>11.5f}{abs(num - ana):>14.2e}")
print(f"\nworst disagreement: {worst:.2e}  -> the analytic form is correct")

# Gradient descent: subtract the slope, scaled by a learning rate.
w, lr = 8.0, 0.2
print("\nstep    w        L(w)     slope")
for step in range(7):
    g = analytic_slope(w)
    print(f"{step:>4} {w:>7.4f} {loss(w):>9.4f} {g:>+8.3f}")
    w -= lr * g
print(f"converging on w = 3, where the loss is its minimum of 2")

The sign is the whole message. A **positive** slope means "moving right increases the loss", so to *decrease* the loss you move left; a negative slope means the opposite. In both cases the rule is the same, and every optimiser in this notebook uses it:

$$
w \leftarrow w - \eta \cdot \frac{dL}{dw}
$$

Keep `numeric_slope` in mind — we use it again on the DPO gradient, where the hand-derived form is genuinely easy to get wrong.

## 2. Explore or exploit: three slot machines

The smallest possible RL problem. Three arms pay 1 or 0 with unknown probabilities; you get 2000 pulls. Pull the arm you currently believe is best and you may never discover a better one. Pull at random and you waste pulls on arms you already know are bad.

**Epsilon-greedy** is the simplest resolution: with probability $\varepsilon$ pick uniformly at random, otherwise pick the current best. The score to watch is **regret** — the gap between what you earned and what a perfect player would have earned. Read the curves as *slopes*, not heights: a flat curve means you are currently playing optimally; a straight line means you are losing at a constant rate and learning nothing.

In [ ]:
import matplotlib.pyplot as plt

TRUE_MEANS = np.array([0.30, 0.55, 0.50])    # arm 1 is best; arm 2 is a near-miss
BEST = TRUE_MEANS.max()

def run_bandit(epsilon, n_steps=2000, seed=0):
    """Epsilon-greedy over 3 Bernoulli arms. Returns estimates, pulls, regret."""
    rng = np.random.default_rng(seed)
    Q = np.zeros(3)             # value estimate for each arm (starts at zero)
    N = np.zeros(3)             # how many times each arm was pulled
    regret, running = np.zeros(n_steps), 0.0
    for t in range(n_steps):
        if rng.random() < epsilon:
            a = int(rng.integers(3))            # explore: pick uniformly
        else:
            a = int(Q.argmax())                 # exploit: pick the current best
        reward = float(rng.random() < TRUE_MEANS[a])     # pays 1 or 0
        N[a] += 1
        Q[a] += (reward - Q[a]) / N[a]          # running mean, no list needed
        running += BEST - TRUE_MEANS[a]         # what this pull cost us
        regret[t] = running
    return Q, N, regret

print(f"true means: {TRUE_MEANS}   (arm 1 is optimal)\n")
print(f"{'eps':>5}{'Q0':>7}{'Q1':>7}{'Q2':>7}{'pulls of arm 1':>18}{'regret':>9}")
curves = {}
for eps in [0.0, 0.02, 0.10, 0.50]:
    Q, N, regret = run_bandit(eps)
    curves[eps] = regret
    print(f"{eps:>5.2f}{Q[0]:>7.3f}{Q[1]:>7.3f}{Q[2]:>7.3f}"
          f"{int(N[1]):>14}/2000{regret[-1]:>9.1f}")

fig, ax = plt.subplots(figsize=(7.0, 3.6))
for eps, regret in curves.items():
    ax.plot(regret, label=f"epsilon = {eps}")
ax.set_xlabel("pull number")
ax.set_ylabel("cumulative regret")
ax.set_title("Exploration is an investment: it costs early and pays later")
ax.legend()
fig.tight_layout()

The $\varepsilon = 0$ line is the lesson. It is perfectly straight, because a purely greedy agent starting from $Q = [0, 0, 0]$ picks arm 0, and arm 0's estimate can only fall at or below zero while the others stay exactly zero — so it never has a reason to try anything else. Look at the printed estimates: `Q1` and `Q2` are still exactly `0.000`, because those arms were **never pulled at all**. A policy that never explores can only confirm what it already believes.

$\varepsilon = 0.02$ shows the subtler failure: it escapes arm 0 but settles on arm 2, whose true mean of 0.50 is barely below arm 1's 0.55, and it never gathers enough evidence to tell them apart. Too little exploration does not merely slow you down — it leaves you confidently committed to a near-miss.

$\varepsilon = 0.50$ shows the opposite excess. Its $Q$ estimates are the most *accurate* of the four, because every arm gets pulled hundreds of times, and yet it accumulates far more regret than $\varepsilon = 0.10$, because it spends half of every session throwing away what it knows. Knowing the right answer earns nothing if you do not use it.

This trade-off never goes away. In an LLM RL run, temperature and top-p are your $\varepsilon$: too low and the model only regenerates what it already does, so there is nothing new to score; too high and you are scoring nonsense.

### Micro-exercise: optimistic initialisation

Exploration does not have to be random. Start every estimate *too high* — `Q = np.full(3, 1.0)` — and pure greedy will try each arm at least once, because any untried arm looks better than a tried one. Copy `run_bandit`, add an `init` argument, and compare the final regret of $(\varepsilon=0, \text{init}=1.0)$ against $(\varepsilon=0.10, \text{init}=0.0)$.

In [ ]:
def run_bandit_init(epsilon, init=0.0, n_steps=2000, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.full(3, init)
    N = np.zeros(3)
    running = 0.0
    # your code here: the same loop as run_bandit, returning the final regret
    return running

# Uncomment to test:
# print("greedy + optimistic:", run_bandit_init(0.0, init=1.0))
# print("eps=0.10, no bonus :", run_bandit_init(0.10, init=0.0))

## 3. REINFORCE: learning a policy from reward alone

A **policy** is a distribution over actions. We parameterise it with logits and a softmax, so the parameters are unconstrained while the output is always a valid distribution. **REINFORCE** is the whole idea of policy gradients in one line: sample an action, see the reward, and push the log-probability of that action up in proportion to the reward.

$$
\theta \leftarrow \theta + \eta \, r \, \nabla_\theta \log \pi_\theta(a)
$$

For a softmax policy, $\nabla_\theta \log \pi_\theta(a)$ is exactly `onehot(a) - p`. Here it is on a **contextual bandit**: a customer message arrives in one of two moods, the policy picks one of three replies, and a noisy reward comes back. Six parameters, and you can watch every one of them move.

In [ ]:
CONTEXTS = ["angry", "confused"]
ACTIONS = ["apologise", "explain", "joke"]
REWARD = np.array([[1.0, 0.3, -1.0],       # what an angry customer wants
                   [0.1, 1.0, -0.5]])      # what a confused customer wants

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def reinforce(n_steps=600, lr=0.2, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((2, 3))               # 6 parameters, all starting equal
    rewards, snapshots = [], {}
    for t in range(n_steps):
        if t in (0, 50, 200, n_steps - 1):
            snapshots[t] = np.array([softmax(theta[0]), softmax(theta[1])])
        c = int(rng.integers(2))                     # the world picks a context
        p = softmax(theta[c])                        # the policy's distribution
        a = int(rng.choice(3, p=p))                  # sample an action (explore!)
        r = REWARD[c, a] + 0.3 * rng.normal()        # noisy reward from the world

        onehot = np.zeros(3)
        onehot[a] = 1.0
        grad_logp = onehot - p                       # d log pi(a) / d theta
        theta[c] += lr * r * grad_logp               # REINFORCE: ascend r * grad
        rewards.append(r)
    return theta, np.array(rewards), snapshots

theta_pg, rewards, snapshots = reinforce()
print("policy probabilities over", "/".join(ACTIONS), "\n")
for t, snap in snapshots.items():
    left = " ".join(f"{v:.2f}" for v in snap[0])
    right = " ".join(f"{v:.2f}" for v in snap[1])
    print(f"step {t:>4}   angry: [{left}]   confused: [{right}]")
print(f"\nmean reward, first 100 steps: {rewards[:100].mean():.3f}")
print(f"mean reward, last  100 steps: {rewards[-100:].mean():.3f}")

smooth = np.convolve(rewards, np.ones(40) / 40, mode="valid")
fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.plot(smooth)
ax.set_xlabel("step")
ax.set_ylabel("reward (40-step moving average)")
ax.set_title("REINFORCE on a two-context, three-action bandit")
fig.tight_layout()

The policy starts uniform — every reply equally likely in every mood — and ends essentially deterministic *and context-dependent*: apologise to the angry customer, explain to the confused one. Nobody ever told it which reply was correct. It tried all three, the world scored them, and the log-probabilities moved.

**What is toy, what is faithful.** Toy: six parameters, three actions, a reward table instead of a human. Faithful: `theta[c] += lr * r * (onehot - p)` *is* the REINFORCE gradient for a softmax policy, and the sample-then-score structure is exactly what an LLM RL run does — it just samples a 500-token trajectory instead of a single action.

## 4. PPO's clip, drawn rather than argued

REINFORCE throws each sample away after one update, which is wasteful when generating the sample cost a GPU-second. **PPO** reuses a batch by tracking the **importance ratio** $r = \pi_\theta(a) / \pi_{\text{old}}(a)$ — how much more likely the current policy is to take that action than the policy that generated it. Reuse is only safe while $r$ stays near 1, so PPO clips:

$$
L^{\text{CLIP}} = \min\bigl(r A,\; \operatorname{clip}(r, 1-\epsilon, 1+\epsilon) A\bigr)
$$

with $\epsilon$ typically 0.1–0.2, and $A$ the **advantage** (how much better than average this action was). The `min` is the clever part, and it is much easier to see than to argue about.

In [ ]:
EPS = 0.2

def clipped_objective(ratio, advantage, eps=EPS):
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped)      # min, for both signs of A

ratios = np.linspace(0.0, 2.0, 400)
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4), sharey=True)
for ax, A in zip(axes, [+1.0, -1.0]):
    ax.plot(ratios, ratios * A, "--", color="0.7", label="unclipped  r*A")
    ax.plot(ratios, clipped_objective(ratios, A), lw=2, label="PPO objective")
    ax.axvline(1 - EPS, color="0.85", zorder=0)
    ax.axvline(1 + EPS, color="0.85", zorder=0)
    ax.set_xlabel("probability ratio  r")
    ax.set_title(f"advantage A = {A:+.0f}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("objective (higher is better)")
fig.tight_layout()

def has_gradient(ratio, advantage, eps=EPS):
    """The min picks the constant branch exactly when the ratio overshoots."""
    frozen = (advantage > 0 and ratio > 1 + eps) or (advantage < 0 and ratio < 1 - eps)
    return "zero" if frozen else "live"

print(f"{'ratio':>7}{'A=+1 obj':>11}{'grad':>7}{'A=-1 obj':>11}{'grad':>7}")
for r in [0.5, 0.79, 1.0, 1.21, 1.5]:
    print(f"{r:>7.2f}{clipped_objective(r, 1.0):>11.3f}{has_gradient(r, 1.0):>7}"
          f"{clipped_objective(r, -1.0):>11.3f}{has_gradient(r, -1.0):>7}")

Read the left panel ($A > 0$, a good action). The objective rises with the ratio until $1 + \epsilon$, then goes flat: making a good action *even more* likely stops paying, the gradient becomes zero, and the update stops. The right panel ($A < 0$, a bad action) is the mirror image — you get credit for making a bad action less likely, up to $1 - \epsilon$.

The asymmetry is what the `min` buys. On the left panel the objective still *falls* below $1 - \epsilon$: if the ratio went the wrong way on a good action, the gradient is alive and pulls it back. The clip only removes the incentive to keep pushing in the direction you were already going too far in. It never removes the incentive to come back.

## 5. DPO: the reward model was never needed

For the KL-regularised RLHF objective, the best possible policy has a closed form, $\pi^{*}(y \mid x) \propto \pi_{\text{ref}}(y \mid x)\exp(r(x,y)/\beta)$. Everyone treated that as a *goal*. Rafailov et al. (2023) rearranged it instead:

$$
r(x, y) = \beta \log \frac{\pi^{*}(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)
$$

**Any policy implicitly defines a reward function** — $\beta$ times its log-ratio against a frozen reference. The intractable $\log Z(x)$ cancels, because preference data compares two responses *to the same prompt* and only their difference matters. Substituting into Bradley-Terry gives the DPO loss:

$$
\mathcal{L}_{\text{DPO}} = -\,\mathbb{E}\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \beta \log \frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\right)\right]
$$

Look at what is left: a fixed dataset, a sigmoid, a cross-entropy. **No sampling, no reward model, no value model, no rollout.** DPO is a supervised loop that happens to optimise an RL objective — which is why it took over.

Two prompts, five candidate responses each (so the policy is ten logits), a frozen reference that favours what a badly-tuned model favours, and eight hand-written preferences. The cell derives the analytic gradient and then checks it with `numeric_slope`'s two-sided trick.

In [ ]:
PROMPTS = ["explain recursion", "apologise for the outage"]
RESPONSES = [
    ["one-line definition", "worked example", "wall of jargon",
     "flatly wrong", "off-topic ramble"],
    ["just 'sorry'", "sorry + cause + fix", "blames the user",
     "corporate word salad", "off-topic ramble"],
]
# The frozen reference: what our SFT model does before any preference training.
REF_LOGITS = np.array([[0.5, 0.0, 1.0, -0.5, 0.0],
                       [0.3, 0.2, -0.5, 1.0, 0.0]])
# Eight hand-made preferences: (prompt, winner index, loser index).
PAIRS = [(0, 1, 0), (0, 1, 2), (0, 0, 3), (0, 1, 3),
         (1, 1, 0), (1, 1, 2), (1, 0, 3), (1, 1, 3)]
BETA = 0.1

def log_softmax(z):
    m = z.max()
    return z - m - np.log(np.exp(z - m).sum())

def margins(theta, beta=BETA):
    """The implicit-reward margin beta*(r_hat(y_w) - r_hat(y_l)) for each pair."""
    out = []
    for p, w, l in PAIRS:
        lp, lref = log_softmax(theta[p]), log_softmax(REF_LOGITS[p])
        out.append(beta * ((lp[w] - lref[w]) - (lp[l] - lref[l])))
    return np.array(out)

def dpo_loss(theta, beta=BETA):
    s = margins(theta, beta)
    return float(np.mean(-np.log(1.0 / (1.0 + np.exp(-s)))))    # -log sigmoid(s)

def dpo_grad(theta, beta=BETA):
    """Analytic: -(1 - sigma(margin)) * beta * (push winner up, pull loser down)."""
    g = np.zeros_like(theta)
    for (p, w, l), s in zip(PAIRS, margins(theta, beta)):
        sigma = 1.0 / (1.0 + np.exp(-s))
        coef = -(1.0 - sigma) * beta
        g[p, w] += coef
        g[p, l] -= coef
    return g / len(PAIRS)

def finite_difference_grad(theta, h=1e-6):
    g = np.zeros_like(theta)
    for i in range(theta.shape[0]):
        for j in range(theta.shape[1]):
            up, down = theta.copy(), theta.copy()
            up[i, j] += h
            down[i, j] -= h
            g[i, j] = (dpo_loss(up) - dpo_loss(down)) / (2 * h)
    return g

theta = REF_LOGITS.copy()          # DPO always starts the policy at the reference
print(f"starting loss: {dpo_loss(theta):.6f}   (= -log 0.5 = {np.log(2):.6f}, "
      f"because policy == reference makes every margin exactly 0)\n")

analytic, numeric = dpo_grad(theta), finite_difference_grad(theta)
print("analytic gradient (prompt 0):", np.round(analytic[0], 6))
print("numeric  gradient (prompt 0):", np.round(numeric[0], 6))
print(f"worst disagreement: {np.abs(analytic - numeric).max():.2e}")
assert np.abs(analytic - numeric).max() < 1e-6
print("gradient check passed")

Read the gradient numbers as well as the check, because they are the algorithm's intent in five figures. `worked example` (index 1) has the most negative entry, so descent will raise it hardest — it wins three of prompt 0's four pairs. `flatly wrong` (index 3) loses two pairs and gets twice the positive value of `wall of jargon`, which loses one. And the two zeros have *different causes*: `off-topic ramble` (index 4) appears in no pair at all, so DPO has nothing to say about it, while `one-line definition` (index 0) wins one pair and loses one, so its two contributions cancel exactly.

Now train.

In [ ]:
theta = REF_LOGITS.copy()
losses, mean_margin = [], []
for step in range(150):
    losses.append(dpo_loss(theta))
    mean_margin.append(float(margins(theta).mean()))
    theta -= 1.0 * dpo_grad(theta)          # plain gradient descent, lr = 1.0
losses.append(dpo_loss(theta))

correct = sum(1 for m in margins(theta) if m > 0)
print(f"loss   {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"margin {mean_margin[0]:.4f} -> {float(margins(theta).mean()):.4f}")
print(f"pairs ranked correctly: {correct}/{len(PAIRS)}")

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.2))
axes[0].plot(losses)
axes[0].set_xlabel("step"); axes[0].set_ylabel("DPO loss"); axes[0].set_title("loss")
axes[1].plot(mean_margin)
axes[1].set_xlabel("step"); axes[1].set_ylabel("mean implicit-reward margin")
axes[1].set_title("margin")
fig.tight_layout()

The loss falls, but nowhere near zero — and if that surprises you, it is the most useful surprise on the page. With $\beta = 0.1$ the loss is $-\log \sigma(0.1 \times \text{log-ratio gap})$, so reaching a loss of 0.2 would need a log-ratio gap of about 15, meaning a policy astronomically far from the reference. **A DPO loss that stalls around 0.5 is normal and often healthy.** Judge the run by the margin and the ranking accuracy — all eight pairs correct — not by the loss reaching zero.

Now the before-and-after, which is what you actually wanted to know.

In [ ]:
def softmax_row(z):
    e = np.exp(z - z.max())
    return e / e.sum()

for p, prompt in enumerate(PROMPTS):
    before, after = softmax_row(REF_LOGITS[p]), softmax_row(theta[p])
    implicit = BETA * (np.log(after) - np.log(before))
    print(f"\n{prompt!r}")
    print(f"   {'response':<22}{'reference':>11}{'policy':>9}{'change':>10}"
          f"{'implicit r':>13}")
    for j, name in enumerate(RESPONSES[p]):
        print(f"   {name:<22}{before[j]:>11.3f}{after[j]:>9.3f}"
              f"{after[j] - before[j]:>+10.3f}{implicit[j]:>13.3f}")

print(f"\n{'pair':<40}{'d log pi(win)':>15}{'d log pi(lose)':>16}")
for p, w, l in PAIRS:
    b, a = log_softmax(REF_LOGITS[p]), log_softmax(theta[p])
    label = f"{RESPONSES[p][w][:16]} > {RESPONSES[p][l][:16]}"
    print(f"{label:<40}{a[w] - b[w]:>+15.3f}{a[l] - b[l]:>+16.3f}")

Read the first table by row: probability mass moved off the reference's favourites — `wall of jargon` and `corporate word salad` — and onto the responses the preferences endorse. The `implicit r` column is the reward function DPO learned **without ever building one**: $\beta$ times the log-ratio, positive for responses the policy now prefers over the reference and negative for the rest.

Now read the second table, which is DPO's most-cited failure mode made visible. Find the row `one-line definition > flatly wrong`: the *preferred* response had its log-probability driven **down**. Nothing in the objective says the winner's probability must rise — it only constrains the *difference*, and this response is also the loser in `worked example > one-line definition`, where the downward pull was stronger. DPO satisfied both comparisons correctly and made both sides of one pair less likely while doing it. On a real model this is how DPO runs drift toward shorter, blander text: the mass has to go somewhere, and the loss never said where. Log the absolute log-probabilities, not just the margin.

**What is toy, what is faithful.** Toy: five candidate responses instead of a vocabulary raised to the sequence length, ten parameters instead of seven billion, eight pairs instead of a hundred thousand, and a "log-probability of a response" that is one softmax entry rather than a sum over tokens. Faithful: `dpo_loss` and `dpo_grad` are the DPO objective and its exact gradient, and initialising the policy at the reference is what every implementation does.

$\beta$ converts log-ratio into reward, so it sets how far the policy is willing to travel. The sweep below trains the same data at four values.

In [ ]:
def kl_from_reference(th):
    total = 0.0
    for p in range(len(PROMPTS)):
        a, b = softmax_row(th[p]), softmax_row(REF_LOGITS[p])
        total += float(np.sum(a * np.log(a / b)))
    return total / len(PROMPTS)

print(f"{'beta':>6}{'log gap for 1 nat':>20}{'= probability ratio':>22}")
for beta in [2.0, 0.5, 0.1, 0.02]:
    print(f"{beta:>6.2f}{1.0 / beta:>20.1f}{np.exp(1.0 / beta):>22.3g}")

print("\n150 steps at lr = 1.0, same data, different beta:")
print(f"{'beta':>6}{'final loss':>13}{'KL from ref':>14}{'P(worked example)':>20}")
for beta in [0.02, 0.1, 0.5, 2.0]:
    th = REF_LOGITS.copy()
    for _ in range(150):
        th -= 1.0 * dpo_grad(th, beta)
    print(f"{beta:>6.2f}{dpo_loss(th, beta):>13.4f}{kl_from_reference(th):>14.3f}"
          f"{softmax_row(th[0])[1]:>20.3f}")

$\beta = 0.02$ is **too small**: the gradient is proportional to $\beta$, so after 150 steps almost nothing has moved. It would get there eventually — and "eventually" is the problem, because by then the policy would sit an enormous distance from the reference, which the first table prices in probability ratios.

$\beta = 2.0$ is **too large** in the opposite way: the loss collapses almost immediately, and once $\sigma(\text{margin}) \approx 1$ the gradient factor $(1 - \sigma)$ is essentially zero, so training simply stops. The policy freezes wherever the first few steps put it, having learned nothing from the last 140. $\beta = 0.1$ is what most implementations default to, and the middle of that spread is why.

## 6. Reward hacking, made visible

Every method above optimises a number. Where does the number come from? For open-ended tasks, from a **reward model** trained on human comparisons — and humans are not neutral instruments. One bias is famous: longer answers *look* more thorough to a tired annotator.

So we simulate it honestly. Twelve responses, described by four features. `W_TRUE` is what we actually want, with length weighted **zero**. `W_ANNOTATOR` is how our labellers behave: identical, except they also reward length. We fit a Bradley-Terry reward model on the annotators' labels — the model never sees `W_TRUE` — and then hand that reward model to an optimiser.

In [ ]:
FEATURES = ["correct", "example", "polite", "length"]
NAMES = ["terse but correct", "correct + example", "correct, blunt",
         "correct + polite", "hedging word salad", "wrong but polite",
         "wrong and terse", "correct + example, long", "correct, no example",
         "off-topic but long", "example only", "padded example, no answer"]
X = np.array([[1, 0, 0, 0], [1, 1, 1, 1], [1, 0, 0, 1], [1, 0, 1, 1],
              [0, 0, 1, 5], [0, 0, 1, 2], [0, 0, 0, 0], [1, 1, 0, 2],
              [1, 0, 1, 0], [0, 0, 0, 3], [0, 1, 0, 1], [0, 1, 1, 3]], dtype=float)

W_TRUE = np.array([2.0, 1.0, 0.6, 0.0])        # length is genuinely irrelevant
W_ANNOTATOR = np.array([2.0, 1.0, 0.6, 1.0])   # ...but longer looks better

true_quality = X @ W_TRUE
annotator_score = X @ W_ANNOTATOR
DATA = [(i, j) for i in range(len(X)) for j in range(len(X))
        if annotator_score[i] > annotator_score[j] + 1e-9]     # (winner, loser)

def rm_loss_and_grad(w, data):
    """Bradley-Terry: P(i beats j) = sigmoid(w.x_i - w.x_j)."""
    loss, grad = 0.0, np.zeros_like(w)
    for i, j in data:
        d = X[i] - X[j]
        s = 1.0 / (1.0 + np.exp(-(w @ d)))
        loss -= np.log(s + 1e-12)
        grad -= (1.0 - s) * d
    return loss / len(data), grad / len(data)

w = np.zeros(4)
for _ in range(1500):
    _, g = rm_loss_and_grad(w, DATA)
    w -= 1.0 * g
w = w / np.linalg.norm(w) * np.linalg.norm(W_ANNOTATOR)        # rescale for reading

print(f"{len(DATA)} preference pairs labelled by the annotator model\n")
print(f"{'feature':<10}{'W_TRUE':>9}{'W_ANNOTATOR':>13}{'reward model learned':>22}")
for k, f in enumerate(FEATURES):
    print(f"{f:<10}{W_TRUE[k]:>9.2f}{W_ANNOTATOR[k]:>13.2f}{w[k]:>22.2f}")
print("\nThe reward model learned a POSITIVE weight on length, which truth says is 0.")
print("It is not wrong about the data. It is wrong about what we wanted.")

Now let the optimiser loose on that reward model and watch two numbers: the **proxy** it can see, and the **true quality** it cannot.

In [ ]:
proxy_reward = X @ w

def optimise(score, steps=200, lr=0.05):
    """Gradient ascent of a softmax policy on expected score."""
    th = np.zeros(len(X))
    got_proxy, got_true, got_len = [], [], []
    for _ in range(steps):
        p = softmax_row(th)
        got_proxy.append(float(p @ score))
        got_true.append(float(p @ true_quality))
        got_len.append(float(p @ X[:, 3]))
        th += lr * p * (score - p @ score)          # d E[score] / d theta
    return np.array(got_proxy), np.array(got_true), np.array(got_len), softmax_row(th)

pr, tr, ln, policy = optimise(proxy_reward)
peak = int(tr.argmax())
print(f"proxy reward   {pr[0]:.3f} -> {pr[-1]:.3f}   (up {pr[-1] - pr[0]:+.3f})")
print(f"TRUE quality   {tr[0]:.3f} -> {tr[-1]:.3f}   ({tr[-1] - tr[0]:+.3f}, "
      f"after peaking at {tr.max():.3f} on step {peak})")
print(f"mean length    {ln[0]:.2f} -> {ln[-1]:.2f}")
print(f"\nthe policy converged on {NAMES[int(policy.argmax())]!r} "
      f"(probability {policy.max():.3f})")
print(f"   its reward-model score is {proxy_reward[int(policy.argmax())]:.2f} "
      f"(the highest) and its true quality is "
      f"{true_quality[int(policy.argmax())]:.2f}")
print(f"   the genuinely best response, {NAMES[int(true_quality.argmax())]!r}, "
      f"scores {true_quality.max():.2f} on truth")

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(pr, label="proxy reward (what we optimise)")
ax.plot(tr, label="true quality (what we wanted)")
ax.plot(ln, "--", label="mean response length")
ax.axvline(peak, color="0.8", zorder=0)
ax.set_xlabel("optimisation step")
ax.set_ylabel("value")
ax.set_title("Goodhart's law, plotted")
ax.legend(fontsize=8)
fig.tight_layout()

There it is. The proxy climbs steadily and never stops looking healthy. True quality rises for the first forty-odd steps — a small but real improvement — and then *falls*, ending well below where it started, while mean response length nearly triples. The optimiser found the long, polite, incorrect answer that the reward model was taught to love, and it is not cheating: it is maximising exactly what we gave it.

This is **Goodhart's law**: when a measure becomes a target, it ceases to be a good measure. And the vertical line at the true-quality peak is the part that should worry you: if you had stopped there you would have shipped a genuine improvement, and **the reward curve gives you no signal at all about where that line is** — it looks identical either side. The only trustworthy stopping signal is an independent evaluation the optimiser cannot see.

Reward hacking is not a bug in the optimiser. The optimiser did exactly what it was told, perfectly. Every reward-hacking story reduces to a specification that was subtly wrong and an optimiser that was fully competent.

## 7. pass@k, and the estimator that makes it honest

Code benchmarks do not compare strings, they run the program — and a model sampling at nonzero temperature is a *distribution* over programs, so "did it solve the problem" is not a yes/no question. The standard answer is **pass@k**: the probability that at least one of $k$ independent samples passes the tests.

The naive estimate draws exactly $k$ samples and checks. It works, and it is extremely noisy. The estimator introduced with HumanEval draws $n > k$ samples, counts how many pass, and computes the chance that a random $k$-subset misses every correct one:

$$
\text{pass@}k = \mathbb{E}_{\text{tasks}}\left[1 - \frac{\binom{n - c}{k}}{\binom{n}{k}}\right]
$$

The tempting shortcut — treat $c/n$ as $p$ and use $1 - (1-p)^k$ — is *biased*. The cell prints one eval run per task, then the benchmark means, and finally repeats the whole eval 300 times at four values of $n$ so the bias and the noise can be read off separately.

In [ ]:
from math import comb

# Eight tasks with different true per-sample success rates. In a real eval you
# never see these numbers; here we do, so the estimators can be checked.
P_TRUE = np.array([0.95, 0.70, 0.55, 0.40, 0.25, 0.10, 0.05, 0.00])
N_SAMPLES = 20

def pass_at_k(n, c, k):
    """Unbiased pass@k for ONE task: n samples drawn, c of them correct."""
    if n - c < k:                     # every k-subset contains a correct one
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def plug_in(n, c, k):
    """The tempting shortcut: treat c/n as p and assume independence."""
    return 1.0 - (1.0 - c / n) ** k

counts = np.random.default_rng(0).binomial(N_SAMPLES, P_TRUE)
print(f"per-task, n = {N_SAMPLES}")
print(f"{'p_true':>7}{'c':>4}{'pass@1':>9}{'pass@5':>9}{'pass@10':>9}")
for p, c in zip(P_TRUE, counts):
    print(f"{p:>7.2f}{c:>4}" + "".join(f"{pass_at_k(N_SAMPLES, c, k):>9.3f}"
                                       for k in (1, 5, 10)))

print("\nbenchmark score = the mean of that column")
print(f"{'k':>3}{'unbiased':>11}{'plug-in':>10}{'truth':>9}")
for k in (1, 5, 10):
    unb = float(np.mean([pass_at_k(N_SAMPLES, c, k) for c in counts]))
    plg = float(np.mean([plug_in(N_SAMPLES, c, k) for c in counts]))
    tru = float(np.mean(1 - (1 - P_TRUE) ** k))
    print(f"{k:>3}{unb:>11.3f}{plg:>10.3f}{tru:>9.3f}")

print("\nsame eval repeated 300 times, estimating pass@5 "
      f"(truth {float(np.mean(1 - (1 - P_TRUE) ** 5)):.3f})")
print(f"{'n':>5}{'unbiased mean':>15}{'std':>8}{'plug-in mean':>14}{'std':>8}")
for n in (5, 10, 20, 100):
    r = np.random.default_rng(1)
    u, g = [], []
    for _ in range(300):
        c = r.binomial(n, P_TRUE)
        u.append(np.mean([pass_at_k(n, ci, 5) for ci in c]))
        g.append(np.mean([plug_in(n, ci, 5) for ci in c]))
    print(f"{n:>5}{np.mean(u):>15.3f}{np.std(u):>8.3f}"
          f"{np.mean(g):>14.3f}{np.std(g):>8.3f}")

Two lessons in the last table. The unbiased estimator's mean sits on the truth at every $n$, while its standard deviation shrinks as $n$ grows — that is exactly what "draw more than $k$" buys you: less noise, same expected answer. The plug-in shortcut is *systematically biased low*, badly so at $n = 5$, and it only creeps toward the truth as $n$ gets large. Its error is not noise you can average away by re-running the eval; it is there in every run.

The practical rule: report pass@1 and pass@10 from the same $n = 20$ or $n = 100$ samples, using the unbiased estimator, and say what $n$ was. A pass@10 computed from ten samples is a single coin flip dressed as a benchmark.

## 8. An eval harness, and the interval nobody reports

An eval harness is four parts: a **task** (input plus expected answer plus how to score it), a **model**, a **runner**, and a **report**. Keeping the scorer *with* the task is the design decision that matters — different questions need different notions of correct, and a single global `==` quietly fails every one of them.

In [ ]:
import re

class Task:
    def __init__(self, task_id, prompt, expected, scorer):
        self.id, self.prompt, self.expected, self.scorer = (
            task_id, prompt, expected, scorer)

def score_exact(output, task):
    return float(output.strip() == task.expected)

def score_contains(output, task):
    return float(task.expected.lower() in output.lower())

def score_regex(output, task):
    return float(re.search(task.expected, output) is not None)

def score_numeric(output, task):
    """The LAST number in the output, accepted within 1%.

    'the last number' is a convention, and conventions are where evals leak:
    the first number in '7 + 5 = 12' is 7, which would score this wrong.
    """
    found = re.findall(r"-?\d+(?:\.\d+)?", output)
    if not found:
        return 0.0
    return float(abs(float(found[-1]) - float(task.expected))
                 <= 0.01 * abs(float(task.expected)))

TASKS = [
    Task("cap-fr", "Capital of France?", "Paris", score_contains),
    Task("cap-jp", "Capital of Japan?", "Tokyo", score_contains),
    Task("sum-12", "What is 7 + 5?", "12", score_numeric),
    Task("div-pi", "Divide 22 by 7.", "3.142857", score_numeric),
    Task("yesno", "Is 17 prime? Answer yes or no.", "yes", score_exact),
    Task("json", "Give me an empty JSON object.", r"^\s*\{\s*\}\s*$", score_regex),
]

class FakeLLM:
    """A scripted model: answers keyed by task id, with a fallback."""
    def __init__(self, name, answers, fallback="I am not sure."):
        self.name, self.answers, self.fallback = name, answers, fallback

    def __call__(self, task):
        return self.answers.get(task.id, self.fallback)

STRONG = FakeLLM("strong-1", {
    "cap-fr": "The capital of France is Paris.",
    "cap-jp": "Tokyo.",
    "sum-12": "7 + 5 = 12",
    "div-pi": "22 / 7 = 3.142857142857143",
    "yesno": "yes",
    "json": "{}"})
WEAK = FakeLLM("weak-1", {
    "cap-fr": "Paris, I think.",
    "cap-jp": "Kyoto was the old capital.",
    "sum-12": "The answer is 13.",
    "div-pi": "about 3.14",
    "yesno": "Yes, 17 is prime.",          # right idea, wrong FORMAT
    "json": "Sure! Here you go: {}"})

def run_eval(model, tasks):
    return [(t, model(t), t.scorer(model(t), t)) for t in tasks]

def report(model, results):
    print(f"model {model.name!r}")
    for task, output, s in results:
        print(f"  {task.id:<7} {'PASS' if s else 'FAIL'}  "
              f"{task.scorer.__name__:<14} {output[:38]!r}")
    print(f"  score: {sum(s for _, _, s in results):.0f}/{len(results)}\n")

for m in (STRONG, WEAK):
    report(m, run_eval(m, TASKS))

Look at `weak-1` on `yesno` and `json`. Both answers are *right* and both are scored wrong, because `score_exact` and a strict regex demand a format the model did not follow. That is not a bug to paper over — it is the single most common reason two people measure the same model and disagree. Either the format is part of the task (then say so in the prompt and score it strictly), or it is not (then use a lenient scorer). Publishing a number without publishing the scorer means nothing.

Then the statistics. Here is the number that should change how you read every eval result: on 50 tasks a 95% confidence interval on accuracy is about **25 points wide**, and most reported improvements are far smaller than that. The **percentile bootstrap** is how you get that interval with no distributional assumptions — resample *tasks* with replacement, re-average, take percentiles.

In [ ]:
N_TASKS = 50

# 1. The same model on the same suite, five runs at temperature > 0.
rng = np.random.default_rng(7)
p_task = rng.uniform(0.15, 0.95, size=N_TASKS)
runs = [rng.binomial(1, p_task).mean() for _ in range(5)]
print("same model, same tasks, temperature > 0:")
print("  run scores: " + "  ".join(f"{s:.1%}" for s in runs))
print(f"  spread {max(runs) - min(runs):.1%} wide, with nothing changed\n")

def boot_ci(x, reps=4000, seed=1, alpha=0.05):
    """Percentile bootstrap: resample TASKS with replacement, re-average."""
    g = np.random.default_rng(seed)
    idx = g.integers(0, len(x), size=(reps, len(x)))
    means = x[idx].mean(axis=1)
    return np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])

r = np.random.default_rng(0)
a = np.zeros(N_TASKS); a[r.choice(N_TASKS, 36, replace=False)] = 1.0   # A: 36/50
b = np.zeros(N_TASKS); b[r.choice(N_TASKS, 38, replace=False)] = 1.0   # B: 38/50
for name, x in (("model A", a), ("model B", b)):
    lo, hi = boot_ci(x)
    print(f"{name}: {x.mean():.1%}   95% CI [{lo:.1%}, {hi:.1%}]   width {hi - lo:.0%}")
print(f"the +{b.mean() - a.mean():.0%} gap sits inside both intervals\n")

# 2. The PAIRED comparison: the same tasks, so the noise cancels.
b2, b8 = a.copy(), a.copy()
zeros = np.flatnonzero(a == 0)
b2[zeros[:2]] = 1.0                           # B = A plus two fixes
b8[zeros[:8]] = 1.0                           # B = A plus eight fixes
print(f"{'scenario':<22}{'A':>5}{'B':>5}{'B-A':>7}{'95% CI on B-A':>20}"
      f"{'verdict':>9}{'  W/L/T':>10}")
for label, other in (("independent errors", b), ("B fixes 2 of A's", b2),
                     ("B fixes 8 of A's", b8)):
    d = other - a
    lo, hi = boot_ci(d)
    w, l = int((d > 0).sum()), int((d < 0).sum())
    print(f"{label:<22}{a.mean():>5.0%}{other.mean():>5.0%}{d.mean():>+7.0%}"
          f"   [{lo:>+5.0%}, {hi:>+5.0%}]{'real' if lo > 0 else 'noise':>9}"
          f"{w:>6}/{l}/{N_TASKS - w - l}")

Three things to take away.

**The same model scores differently on the same suite.** Nothing changed between those five runs except the sampler's luck, and the spread is wider than most claimed improvements.

**Unpaired intervals are wide and mostly useless at this size.** Both single-model intervals are around 25 points wide, and the two-point gap between A and B disappears inside them.

**Pair the comparison and the noise cancels.** Run both models on the *same* tasks and bootstrap the per-task **difference**. The last two rows show why this is the only comparison worth reporting: an improvement that fixes eight of A's failures has an interval clear of zero, while two fixes do not — and the win/loss/tie column tells you *why*, which a single accuracy number never does.

## 9. LLM-as-judge, and the bias you can subtract

When there is no exact answer, one option is to ask a model which of two outputs is better. It is cheap, fast, and correlates decently with human preference — and it has systematic biases that are large enough to manufacture whatever result you were hoping for.

So implement them on purpose. Below, a judge has four biases written as coefficients, and two systems of **equal true quality** are compared 200 times. Ours is merely longer and prettier. Every gap between the true win rate and a measured one is bias.

In [ ]:
rng = np.random.default_rng(0)
N_PAIRS = 200
q_ours = rng.normal(0.0, 1.0, N_PAIRS)          # equal true quality...
q_theirs = rng.normal(0.0, 1.0, N_PAIRS)
len_ours = rng.integers(120, 400, N_PAIRS)      # ...but ours is more verbose
len_theirs = rng.integers(80, 260, N_PAIRS)
TRUE_WIN = float((q_ours > q_theirs).mean())

class BiasedJudge:
    """A FakeLLM judge. Real judges have these tendencies too; the difference
    is that here you can read the coefficients."""
    POSITION = 0.45      # bonus for whichever answer is shown first
    VERBOSITY = 0.005    # bonus per word
    SELF = 0.55          # bonus for text it believes it wrote
    STYLE = 0.60         # bonus for bullet-point formatting

    def utility(self, ans, first):
        return (ans["quality"] + self.VERBOSITY * ans["words"]
                + self.SELF * ans["is_self"] + self.STYLE * ans["bullets"]
                + self.POSITION * first)

    def compare(self, x, y):
        """Returns 'first' or 'second' - the judge only ever sees an order."""
        return "first" if self.utility(x, 1) > self.utility(y, 0) else "second"

JUDGE = BiasedJudge()

def make(i, mine, *, is_self=False, bullets=False):
    return {"quality": q_ours[i] if mine else q_theirs[i],
            "words": int(len_ours[i] if mine else len_theirs[i]),
            "is_self": is_self, "bullets": bullets}

def win_rate(is_self=False, bullets=False, order="both", equal_length=False):
    """Per-pair wins for OUR system, as an array so we can also count flips."""
    wins = np.zeros(N_PAIRS)
    for i in range(N_PAIRS):
        ours, theirs = make(i, True, is_self=is_self, bullets=bullets), make(i, False)
        if equal_length:                           # hand both the same word count
            ours["words"] = theirs["words"] = 200
        if order == "ours-first":
            wins[i] = JUDGE.compare(ours, theirs) == "first"
        elif order == "ours-second":
            wins[i] = JUDGE.compare(theirs, ours) == "second"
        else:                                      # average of both orders
            wins[i] = 0.5 * ((JUDGE.compare(ours, theirs) == "first")
                             + (JUDGE.compare(theirs, ours) == "second"))
    return wins

print(f"true win rate for our system: {TRUE_WIN:.1%}   ({N_PAIRS} pairs)\n")
print(f"{'bias':<12}{'protocol':<34}{'win rate':>9}{'error':>8}")
ROWS = [
    ("position", "ours shown first", win_rate(order="ours-first")),
    ("", "ours shown second", win_rate(order="ours-second")),
    ("", "AVERAGED OVER BOTH ORDERS", win_rate(order="both")),
    ("verbosity", "as generated (ours is longer)", win_rate()),
    ("", "length-matched", win_rate(equal_length=True)),
    ("self-pref", "ours labelled as the judge's own", win_rate(is_self=True)),
    ("formatting", "ours reformatted as bullets", win_rate(bullets=True)),
    ("", "both in plain prose", win_rate(bullets=False)),
]
for bias, protocol, w in ROWS:
    print(f"{bias:<12}{protocol:<34}{w.mean():>9.1%}{w.mean() - TRUE_WIN:>+8.1%}")

flips = win_rate(order="both")
print(f"\norder-flip rate: {(flips == 0.5).mean():.1%} of pairs get a different "
      f"winner when the two answers are swapped")

Read the first three rows together, because they are the fix — and its limit. Showing our answer **first** inflates its win rate by more than twenty points; showing it **second** takes those points straight back; **averaging the two orders** lands between them and removes the presentation artefact entirely. Position bias is the one bias you can delete almost for free, at exactly 2× the judging cost: run every comparison twice with the answers swapped, and count a disagreement as a tie. The order-flip rate tells you how much of your headline number was decided by which answer went first rather than by content.

What the averaged row does **not** do is reach the truth, and that is the honest part. It still sits well above 52.5%, because averaging over orders says nothing about the *other* three biases — our answers really are longer, and the judge really does pay for words. The length-matched row is the one that closes most of the remaining gap. Length-matching is often impossible in practice, so the production version is to regress the judge's score on output length and report the length-controlled effect.

Self-preference is why you should never judge a model with itself. And formatting bias is why "we reformatted our outputs as bullet points" is a change to the *measurement*, not to the product.

The checklist that follows: write the judge prompt as a specification with an explicit rubric; prefer pairwise comparison to absolute scores; **always average both orders**; validate the judge against a few hundred human labels and report the agreement (Cohen's $\kappa$, not raw agreement, because raw agreement flatters any judge on an unbalanced set); and never judge with the model you are shipping.

## What you built

| Piece | The idea in one sentence |
|---|---|
| Finite differences | Check every hand-derived gradient with a two-sided nudge before you trust it. |
| Epsilon-greedy | A policy that never explores can only confirm what it already believes. |
| REINFORCE | Sample, score, push the log-probability of what you did by the reward. |
| PPO clip | Freeze the gradient once the ratio overshoots, but never the way back. |
| DPO | The policy *is* the reward model; the loss is a cross-entropy over pairs. |
| Reward hacking | The proxy keeps rising after true quality starts falling, and looks identical. |
| pass@k | Draw $n > k$ and use the combinatorial estimator, or your number is noise. |
| Bootstrap CI | Pair the comparison, resample tasks, and check whether zero is inside. |
| Judge bias | Average both orders; length-match or regress out length; never self-judge. |

## Try it yourself

Bigger exercises. Every scaffold runs as-is.

### Exercise 1 — Add a baseline to REINFORCE

REINFORCE's gradient estimate is noisy because the reward's *absolute* size scales every update, even though only relative differences matter. Subtracting a **baseline** — the running mean reward — removes most of that variance and changes nothing in expectation. Copy `reinforce`, keep a running mean of the rewards, use `(r - baseline)` in the update, and compare the two reward curves on one labelled plot. Then add a large constant to every entry of `REWARD` and show that the baseline version barely notices while the plain version degrades.

In [ ]:
def reinforce_baseline(n_steps=600, lr=0.2, seed=0, use_baseline=True, offset=0.0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((2, 3))
    baseline, rewards = 0.0, []
    for t in range(n_steps):
        c = int(rng.integers(2))
        p = softmax(theta[c])
        a = int(rng.choice(3, p=p))
        r = REWARD[c, a] + offset + 0.3 * rng.normal()
        # your code here: update `baseline` as a running mean of r, then use
        # advantage = r - baseline if use_baseline else r  in the theta update
        rewards.append(r)
    return np.array(rewards)

# Uncomment to test:
# for off in (0.0, 10.0):
#     for use in (False, True):
#         rs = reinforce_baseline(use_baseline=use, offset=off)
#         print(f"offset={off:>4}  baseline={use!s:<5}  "
#               f"last-100 mean reward {rs[-100:].mean() - off:+.3f}")

### Exercise 2 — GRPO: the group is the baseline

PPO needs a learned value network to supply that baseline. **GRPO** (Group Relative Policy Optimization, introduced with DeepSeekMath, Shao et al. 2024) notices that for a language model you can just sample $G$ responses to the *same* prompt and use their mean as the baseline:

$$
A_i = \frac{r_i - \operatorname{mean}(r_1 \dots r_G)}{\operatorname{std}(r_1 \dots r_G)}
$$

Implement it on three arithmetic prompts with a *verifiable* reward — 1 if the answer is right, 0 if wrong, checked by a rule, so there is nothing to hack. Then count the **zero-variance groups**: when all $G$ responses score the same, every advantage is zero and the whole group produced no gradient. Plot how that count changes as the policy learns, and explain why real GRPO pipelines filter prompts to ones the model solves *sometimes*.

In [ ]:
GRPO_PROMPTS = ["12 * 7 = ?", "sum 1..10 = ?", "17 - 9 = ?"]
CANDIDATES = [["84", "78", "96", "84 (with working)"],
              ["55", "45", "50", "55 (with working)"],
              ["8", "7", "9", "8 (with working)"]]
CORRECT = [{0, 3}, {0, 3}, {0, 3}]          # a rule, not a neural network
INIT = np.array([[0.0, 1.0, 0.5, -0.5],     # the policy starts out mostly wrong
                 [0.0, 0.8, 0.6, -0.4],
                 [0.0, 1.2, 0.3, -0.6]])

def verify(prompt_i, answer_i):
    """The reward function: a checker, not a model. Cannot be hacked."""
    return 1.0 if answer_i in CORRECT[prompt_i] else 0.0

def grpo(n_iters=40, group=8, lr=0.5, seed=0):
    rng = np.random.default_rng(seed)
    th = INIT.copy()
    accuracy, dead_groups = [], []
    for _ in range(n_iters):
        grad, dead = np.zeros_like(th), 0
        for p in range(len(GRPO_PROMPTS)):
            probs = softmax(th[p])
            sampled = rng.choice(4, size=group, p=probs)         # the rollout
            r = np.array([verify(p, int(j)) for j in sampled])
            # your code here: if r.std() is ~0, count a dead group and skip;
            # otherwise adv = (r - r.mean()) / (r.std() + 1e-8) and accumulate
            # grad[p] += mean over the group of adv * (onehot(a) - probs)
        th += lr * grad
        accuracy.append(float(np.mean(
            [sum(softmax(th[p])[j] for j in CORRECT[p])
             for p in range(len(GRPO_PROMPTS))])))
        dead_groups.append(dead)
    return th, np.array(accuracy), np.array(dead_groups)

# Uncomment to test:
# _, acc, dead = grpo()
# print(f"P(correct): {acc[0]:.3f} -> {acc[-1]:.3f}   dead groups: {dead.sum()}")

### Exercise 3 — Find the stopping step without seeing the truth

In section 6 the optimiser sailed past the true-quality peak with no visible signal. The standard mitigation is a **held-out reward model**: fit a second reward model on a *different* slice of the preference data and stop when the two disagree, or when the KL divergence from the starting policy exceeds a budget.

Implement the KL version. Track $\mathrm{KL}(\pi_\theta \Vert \pi_0)$ at every step of `optimise`, find the step where it first exceeds a budget of your choosing, and compare that step with the true-quality peak. Then plot true quality against KL rather than against step number — the shape of that curve is the whole reason KL budgets are used in practice.

In [ ]:
def optimise_with_kl(score, steps=200, lr=0.6):
    th = np.zeros(len(X))
    p0 = softmax_row(th)                      # the uniform starting policy
    kls, trues = [], []
    for _ in range(steps):
        p = softmax_row(th)
        trues.append(float(p @ true_quality))
        # your code here: append float(np.sum(p * np.log(p / p0))) to kls
        th += lr * p * (score - p @ score)
    return np.array(kls), np.array(trues)

# Uncomment to test:
# kls, trues = optimise_with_kl(proxy_reward)
# print("true-quality peak at step", int(trues.argmax()), "with KL", kls[int(trues.argmax())])

### Exercise 4 — How many tasks do you actually need?

You want to detect a 5-point improvement with 80% power. Simulate it: for suite sizes 20, 50, 100, 200, 500, draw many paired runs where model B is genuinely 5 points better, bootstrap the per-task difference each time, and report the fraction of runs whose 95% interval excludes zero. Print the smallest suite size that clears 80%.

The answer will be larger than the suite you were planning to use. That is the point of the exercise.

In [ ]:
def power(n_tasks, gap=0.05, trials=200, seed=3):
    g = np.random.default_rng(seed)
    detected = 0
    for _ in range(trials):
        p = g.uniform(0.15, 0.85, size=n_tasks)
        a_run = g.binomial(1, p).astype(float)
        b_run = g.binomial(1, np.clip(p + gap, 0, 1)).astype(float)
        # your code here: lo, hi = boot_ci(b_run - a_run, reps=800); count lo > 0
    return detected / trials

# Uncomment to test:
# for n in (20, 50, 100, 200, 500):
#     print(f"{n:>4} tasks -> power {power(n):.0%}")